In [ ]:
import glob
import os
from pathlib import Path
from random import choice, choices

import autoroot
import hydra
import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from omegaconf import OmegaConf
from torch import nn

from src.data.lf.lf_datamodule import LF
from src.modules.lit_module import LitModule
from src.modules.models.sequential import SequentialModel
from src.utils import initialize_resolvers
from src.utils.tools import get_rho_lstsq

initialize_resolvers()


def get_lastest_ckpt(path):
    path = Path(path)
    ckpt = max((path).glob("*.ckpt"), key=os.path.getctime)
    return ckpt


path = "./logs/train/2025-01-20_17-07-30/checkpoints"
model = LitModule.load_from_checkpoint(get_lastest_ckpt(path))
model.eval()

data_1 = LF(
    data_dir="",
    N=2000,
    T=512,
    D=1,
    rho_name="impulse",
    rho_param={"alpha": 1},
    input_generator="randn",
    # processing=["smooth_noise"],
    manual_seed=0,
).dataset_construction()[0][:]
inputs = data_1[0].to("cuda:0")
outputs = data_1[1].to("cuda:0")


def compute_gradients(model, data, target, batch_size):
    model.zero_grad()  # Reset gradients
    batch_data = data[:batch_size]
    batch_target = target[:batch_size]
    output = model(batch_data)
    loss = nn.MSELoss()(output, batch_target)
    loss.backward()  # Compute gradients
    gradients = []
    for param in model.parameters():
        if param.grad is not None:
            gradients.append(param.grad.view(-1))  # Flatten gradients
    return torch.cat(gradients)


v2 = compute_gradients(model, inputs, outputs, 2000)
from tqdm import tqdm

res = []
model = model.to("cuda:0")
for i in tqdm(range(1, 2000, 10)):
    v1 = compute_gradients(model, inputs, outputs, i)
    s = torch.dot(v1, v2) / (torch.norm(v1) * torch.norm(v2))
    res.append(s.item())

In [ ]:
cmap = plt.get_cmap(
    "coolwarm"
)  # Use "coolwarm" or you can try "plasma", "inferno", "viridis", etc.

cmap2 = plt.get_cmap("binary")
# Normalize alpha to map the values to colors
norm = plt.Normalize(0, 1)

# Adjust the font size for a two-column LaTeX document
font_size = 22
plt.rcParams.update(
    {
        "font.size": font_size,  # Set default font size (you can increase/decrease as needed)
        "axes.titlesize": font_size,  # Font size for axis titles
        "axes.labelsize": font_size,  # Font size for axis labels
        "xtick.labelsize": font_size,  # Font size for x-tick labels
        "ytick.labelsize": font_size,  # Font size for y-tick labels
        "legend.fontsize": font_size,  # Font size for legend (if used)
        "figure.titlesize": font_size,  # Font size for the figure title (if applicable)
        # 'text.usetex': True,  # Use LaTeX for rendering text
        # 'font.family': 'serif',  # Set font family to serif (common in LaTeX)
        # 'text.latex.preamble': r'\usepackage{amsmath}'
    }
)

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(1, 2000, 10), res, color=cmap(1), linewidth=2)
ax.grid(True, which="both", linestyle="--", linewidth=0.5)
ax.set_yscale("log")  # (Optional) Uncomment if you want y-axis to be logarithmic too

ax.set_xlabel("Batch Size")
ax.set_ylabel("Cosine Similarity")
plt.savefig(f"batch_size.pdf", bbox_inches="tight")  # Save as PDF for LaTeX